In [1]:
import uproot
import pylhe
import numpy as np
import matplotlib.pyplot as plt

In [2]:
file = uproot.open("w+_restframe.root")
tree = file["Events"]

w_E  = tree["w_E"].array()
w_px = tree["w_px"].array()
w_py = tree["w_py"].array()
w_pz = tree["w_pz"].array()

w_list = [[E, px, py, pz] for E,px,py,pz in zip(w_E, w_px, w_py, w_pz)]

In [3]:
e_list = [[E,px,py,pz] for E,px,py,pz in zip(
    tree["e_E"].array(),
    tree["e_px"].array(),
    tree["e_py"].array(),
    tree["e_pz"].array()
)]

ve_list = [[E,px,py,pz] for E,px,py,pz in zip(
    tree["ve_E"].array(),
    tree["ve_px"].array(),
    tree["ve_py"].array(),
    tree["ve_pz"].array()
)]

In [4]:
print(len(w_list))
print(len(e_list))
print(w_list[10])
print(e_list[10])

1000000
1000000
[np.float64(81.78022288291159), np.float64(0.0), np.float64(3.552713678800501e-15), np.float64(0.0)]
[np.float64(40.89011144139801), np.float64(7.36991658068041), np.float64(26.13389213908377), np.float64(30.572949236820797)]


In [5]:
#Renaming
w = w_list
e = e_list
v = ve_list

## Now, to find Polarization Matrix parameters for Spin-1 particle

In [6]:
#First we find costheta and phi for all electrons.
phi_e = []
cos_theta_e = []

for i in range(0, len(e)):
    phi = np.arctan2(e[i][2], e[i][1])
    cos_theta = e[i][3] / np.sqrt(e[i][1]**2 + e[i][2]**2 + e[i][3]**2)
    
    phi_e.append(phi)
    cos_theta_e.append(cos_theta)

In [7]:
print(phi_e[10])
print(cos_theta_e[10])
print(len(phi_e))
print(len(cos_theta_e))

1.295928353227856
0.7476856422807063
1000000
1000000


### Longitudinal polarizations
The angular distribution is integrated first over $2\pi$, then the integral over $\cos \theta$ is done appropriately

Vector asymmetry, $A_3 = \frac{I(0,1) - I(-1,0)}{I(-1,1)}  = \frac{3}{4} \alpha p_z$

In [8]:
I1 = 0
I2 = 0 
for i in range(0, len(cos_theta_e)):
    if cos_theta_e[i] > 0:
        I1 += 1
    else:
        I2 += 1

In [9]:
print(I2)
print(I1)

253470
746530


In [10]:
print(f"Asymmetry, A3 = {(I1 - I2)/len(phi_e)}")

Asymmetry, A3 = 0.49306


Tensor asymmetry, $B_3 = \frac{I(-1, -1/2) - I(-1/2, 0) - I(0 , 1/2) + I(1/2, 1)}{I(-1,1)} = \frac{9}{8 \sqrt{6}} (1 - 3 \delta) T_{zz}$

In [11]:
I1 = 0
I2 = 0
I3 = 0
I4 = 0

for i in range (0, len(cos_theta_e)):
    if cos_theta_e[i] < -0.5:
        I1 += 1
    elif (cos_theta_e[i] > -0.5 and cos_theta_e[i] < 0):
        I2 += 1
    elif (cos_theta_e[i] > 0 and cos_theta_e[i] < 0.5):
        I3 += 1
    else:
        I4 += 1

print(I1 + I2 + I3 + I4)

1000000


In [12]:
print(f"Tensor asymmetry, B3 = {(I1 - I2 - I3 + I4)/len(phi_e)}")

Tensor asymmetry, B3 = 0.105284


### Transverse asymmetries
Here, we integrate over $\cos \theta$ first and then we integrate over $\phi$ on appropriate domains

Vector asymmetry, $A1 = \frac{I(-\pi/2 , \pi/2) - I(\pi/2 , 3\pi/2)}{I(0,2\pi)} = \frac{3}{4} \alpha p_x$

In [13]:
I1 = 0   #(-pi/2, pi/2)
I2 = 0

for i in range(0, len(phi_e)):
    if (phi_e[i] > -np.pi/2 and phi_e[i] < np.pi/2):
        I1 += 1
    else:
        I2 += 1

In [14]:
print(f"Vector asymmetry, A1 = {(I1 -I2)/len(phi_e)}")

Vector asymmetry, A1 = -0.092664


Vector asymmetry, $A2 = \frac{I(0 , \pi) - I(\pi , 2\pi)}{I(0,2\pi)} = \frac{3}{4} \alpha p_y$

In [15]:
I1 = 0   #(0, pi)
I2 = 0

for i in range(0, len(phi_e)):
    if (phi_e[i] > 0 and phi_e[i] < np.pi):
        I1 += 1
    else:
        I2 += 1

In [16]:
print(f"Vector asymmetry, A2 = {(I1 -I2)/len(phi_e)}")

Vector asymmetry, A2 = -0.00102


Tensor asymmetry, $B1 = \frac{I(-\pi/4, \pi/4) - I(\pi/4, 3\pi/4) + I(3\pi/4, 5\pi/4) - I(5\pi/4, 7\pi/4)}{I(0, 2\pi)} = \frac{2}{\pi} (1-3\delta) \frac{T_{xx} - T_{yy}}{\sqrt{6}}$

In [17]:
I1 = 0   #(-pi/4 , pi/4)
I2 = 0   #(pi/4, 3pi/4)
I3 = 0
I4 = 0   #(5pi/4, 7pi/4) equivalent to (-3pi/4, -pi/4)

for i in range(0, len(phi_e)):
    if (phi_e[i] > -np.pi/4 and phi_e[i] < np.pi/4):
        I1 += 1
    elif (phi_e[i] > np.pi/4 and phi_e[i] < 3*np.pi/4):
        I2 += 1
    elif (phi_e[i] > -3*np.pi/4 and phi_e[i] < -np.pi/4):
        I4 += 1
    else:
        I3 += 1

In [18]:
print(f"Tensor asymmetry, B1 = {(I1 - I2 + I3 - I4)/len(phi_e)}")

Tensor asymmetry, B1 = -0.046072


Tensor asymmetry, $B2 = \frac{I(0 , \pi/2) - I(\pi/2, \pi) + I(\pi, 3\pi/2) - I(3\pi/2, 2\pi)}{I(0, 2\pi)} = \frac{2}{\pi} (1-3\delta) \Bigg(\sqrt{\frac{2}{3}} T_{xy}\Bigg)$

In [19]:
I1 = 0   #(0 , pi/2)
I2 = 0   #(pi/2, pi)
I3 = 0   #(pi, 3pi/2) equivalent to (-pi to -pi/2)
I4 = 0   #(3pi/2, 2pi) equivalent to (-pi/2, 0)

for i in range(0, len(phi_e)):
    if (phi_e[i] > 0 and phi_e[i] < np.pi/2):
        I1 += 1
    elif (phi_e[i] > np.pi/2 and phi_e[i] < np.pi):
        I2 += 1
    elif (phi_e[i] > -np.pi/2 and phi_e[i] < 0):
        I4 += 1
    else:
        I3 += 1

In [20]:
print(f"Tensor asymmetry, B2 = {(I1 - I2 + I3 - I4)/len(phi_e)}")

Tensor asymmetry, B2 = 0.001252


## Finding remaining tensorial components $T_{xz}, T_{yz}$

### 1) Txz

The term in the angular distribution has factor $\cos \theta \sin \theta \cos \phi$. Now, since domain of $\cos \theta$ is -1 to 1, the domain of $\theta$ becomes $(0,\pi)$. In this domain $\sin \theta > 0$. Thus the term becomes positive when the product of other two is positive and negative otherwise.

Domain for positive contribution : $$\theta \in (0, \pi/2), \phi \in (0, \pi/2) \cup (3\pi/2, 2\pi),$$  and, $$\theta \in (\pi/2, \pi), \phi \in (\pi/2, 3\pi/2) $$
Domain for negative contribution : $$\theta \in (0, \pi/2), \phi \in (\pi/2, 3\pi/2) ,$$  and, $$\theta \in (\pi/2, \pi), \phi \in (0, \pi/2) \cup (3\pi/2, 2\pi)$$

In [21]:
I1 = 0  #for positive contributions
I2 = 0  #for negative contributions

for i in range(0, len(phi_e)):
    if (cos_theta_e[i]* np.cos(phi_e[i]) > 0):
        I1 += 1
    else:
        I2 += 1
print(f"Tensor asymmetry corresponding to T_xz = {(I1 - I2)/len(phi_e)}")

Tensor asymmetry corresponding to T_xz = -0.0087


### T_yz
Similar to procedure for T_xz,

In [22]:
I1 = 0  #for positive contributions
I2 = 0  #for negative contributions

for i in range(0, len(phi_e)):
    if (cos_theta_e[i]* np.sin(phi_e[i]) > 0):
        I1 += 1
    else:
        I2 += 1
print(f"Tensor asymmetry corresponding to T_yz = {(I1 - I2)/len(phi_e)}")

Tensor asymmetry corresponding to T_yz = -0.000956
